<center><h1 style="margin-bottom: 0px;">Machine Learning I (25/26)</h1></center>
<center><h2 style="margin-top: 0px;">K-Nearest Neighbour Modification and Performance Benchmarking</h2></center>

#### <center> Filipe Zheng (202406753), Maiara Guedes (202407694), Sílvia Pinto (202405988) </center> <br>

In [4]:
import math
import pandas

### **1. Algorithm Selection and Implementation** <br>
<div style="text-align: justify;"><p style="text-indent: 2em;">Having the freedom to choose any of the algorithms we learned in the course lectures, we decided to work with K-Nearest Neighbour. Due to its simplicity, it gives us many options for modification and improvement, and allows us to employ meticulous attention to each detail of this project.
<p style="text-indent: 2em;">To implement K-NN (K-Nearest Neighbour), we must first comprehend how it works. When K-NN receives a new object to classify, it compares it to all the objects in the training set, which are already correctly classified. This comparison is done by measuring the distance between them: how many attributes do they have in common, and what is the difference between the attributes in which they diverge? Then, it chooses the k closest training objects ("neighbours") and checks each of their target classes. The most common class in those k neighbours is the class it will assign to the new object. 
<p style="text-indent: 2em;">The following code shows our implementation. It defines a class which allows us to create and use many individual KNN classifiers - each of them can be "fed" a training set, upon which it will make predictions on new, unseen datasets. It is the unmodified, conventional version of the K-NN algorithm, which we explained above.</div>

In [ ]:
class KNN_Classifier():
    def __init__(self,k=3,*,distance_function=None):                                               # Initializes a new classifier.
        self.k = k                                                                                 # k is the number of neighbours to consider.
        self.df = None                                                                             # The classifier has not yet been "fed" the data - it is empty.
        self.df_target = None                                                                      # The same goes for the target dataframe.
        self.distF = distance_function or KNN_Classifier.distance                                  # Function to calculate the distance.

    def fit(self,data_frame,data_frame_target):                                                    # Fits the dataframes into the classifier.
        self.df = data_frame                                                                       # Predictive attributes.
        self.df_target = data_frame_target                                                         # Target attribute.

    def distance(row1,row2):                                                                       # Default distance function (can be replaced when initializing).
        Sum = 0                                                                             
        for i,j in zip(row1,row2):
            if (isinstance(i,(float,int)) and not(math.isnan(i) or math.isnan(j))): Sum+=abs(i-j)  # For each numeric attribute, add the difference to the sum.
            else: Sum += i!=j                                                                      # For each categorical attribute, add 1 if different, else 0.
        return Sum                                                                                 # Return the distance.

    def predict_row(self,row):                                                                     # Function to predict a new, unclassified object.
        dists = [(i,self.distF(row,row0)) for i,row0 in self.df.iterrows()]                        # Calculate all distances for each of the training set objects.
        knn = [self.df_target.iloc[i[0]] for i in sorted(dists,key=lambda x:x[1])[:self.k]]        # Select the targets of the 3 closest neighbours to our object.
        counter = {}
        for i in knn:
            if not i in counter: counter[i] = 0                                                    # If the class has not been seen yet, add it the group.
            counter[i]+=1                                                                          # Add one vote to the current neighbour's class.
        pred = max(counter,key=lambda x:counter[x])                                                # Predict the class of the object by majority voting.
        return pred                                                                                # Return the predicted value for our object.
            
    def predict(self,df):                                                                          # Function to predict the target class.
        return [self.predict_row(row) for _,row in df.iterrows()]                                  # Returns target predictions for each of the rows in the dataframe.

### **2. Performance Measurement - Before Modification** <br>
<div style="text-align: justify;"><p style="text-indent: 2em;">Now, we will test this original K-NN implementation on the datasets. By measuring various statistics of its performance, we can then decide what kind of modifications we will apply on the algorithm in order to improve those measurements.
<p style="text-indent: 2em;">But first... being provided with 3 different types of data characteristics, we must reflect on which will be the most challenging and interesting to analyse for K-NN. Each of the dataset groups is oriented to a different type of "problematic" data. Respectively, groups 1 , 2 and 3 are significantly affected by noise/outliers, class imbalance and multiclass classification. Considering K-NN's algorithm, we can make the following remarks:

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; **1.** *Because K-NN is a "lazy learner" - it does not have a strong mathematical foundation, since it simply relies on the proximity of data points - it can be very sensitive to noise and outliers. For small values of k, a few outliers can greatly affect the prediction of the algorithm (such as a few elements of one class inside a large agglomeration of elements of another class).* 

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**2.** *Class imbalance is definitely a major influence on K-NN. Since its classification relies on majority voting, if a class is under-represented, it might be chosen much less than it should - because the neighbourhood of our object is statistically more likely to be dominated by the most common class.*

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**3.** *Finally, multiclass classification is also troublesome. With an increase in the number of classes, the decision boundaries become more and more complex and fragmented. Also, areas where many classes overlap become very difficult to classify, since small changes to k will probably make our prediction jump from class to class.*


<p style="text-indent: 2em;">Taking into account the previous points, it seems most likely to us that dataset group 2 will impose a greater challenge to our algorithm (but all of them will definitely have some level of impact on K-NN). A very disproportionate class imbalance might not cause a loss of accuracy; maybe it will become even higher, but that is simply because practically every new object is falling into the larger class, completely disregarding the minority class.
<p style="text-indent: 2em;">Let us test our hypothesis in a tangible way: we will now use our KNN Classifier in each dataset group and analyse various performance measures, to determine which data characteristic truly causes the most damage to our predictions.</div>
